# ③ 검증 — 평가 데이터셋과 골든 근거 라벨
담당: 검증 · 코드: `rag/validation/validation.py`, `rag/validation/evidence.py`

골든 dev 408문항과 RFP 평가 질문을 점검해 `artifacts/eval_set.jsonl`에 저장합니다.
골든 test 943문항은 라벨 품질 보고서에서만 읽고 평가셋에는 넣지 않습니다. 검색·생성·API 호출은 없습니다.


In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, ROOT, load_config, read_jsonl, write_jsonl
from rag.validation.validation import check_eval_items, load_eval_items
from rag.validation.evidence import (
    correspondence_report, document_report, label_evidence, labelled_sample, mapping_report,
)

cfg = load_config()
chunks = read_jsonl(ARTIFACTS / "chunks.jsonl")
raw_dir = ARTIFACTS / "raw"
input_path = ROOT / cfg["paths"]["input"]
package_dir = ROOT / "data/RFP_Preprocessing_Final_v11_Team"


## 근거 항목과 [E] 블록의 대응

`context`에는 인용하지 않은 블록도 있고, `evidence`는 선택한 근거 참조만 담습니다.
순서대로 짝짓지 않습니다. v11 `final_elements.jsonl`의 `element_id`와 `location`으로
각 항목의 본문을 복원합니다. 글은 요소의 `[start:end]`, 표는 해당 행의 칸을 열 순서로 ` | `로 잇습니다.
복원한 본문이 context 블록 전체와 같은지 확인하고 모든 해당 `[E]` 라벨을 보관합니다.
이 좌표는 본문 복원에만 쓰며 현재 청크의 좌표로 변환하지 않습니다.

파서 문서는 설정한 원본 폴더·파일(ZIP은 `artifacts/raw`에 푼 파일)의 SHA-256 → NFC 파일명 → 현재 `source/doc_id`로 연결합니다.
v11은 패키지의 원본 SHA와 현재 `doc_id`·`source`가 모두 일치할 때 연결하므로 raw 폴더가 필요하지 않습니다.
NFC 정규화, 표 파이프를 공백으로 바꾸기, 연속 공백 축약 후 근거 본문이 같은 문서의
청크 `body`에 부분 문자열로 들어가야 합니다. 유일한 청크 ID일 때만 라벨을 붙입니다.
20자 미만은 짧은 수·코드·항목 이름만으로 청크를 확정하지 않도록 `too_short`로 둡니다.
문서 미연결을 먼저 판정하며, 나머지는 `not_found`, `ambiguous`입니다. 유사도 점수는 사용하지 않습니다.


In [ ]:
items = load_eval_items(cfg["validation"]["golden_sets"])
labelled_items = label_evidence(items, chunks, raw_dir, package_dir, input_path)
dev_items = [item for item in labelled_items if item["evidence_source"] == "golden"]
test_input = load_eval_items(["eval/golden/test.jsonl"])
test_items = label_evidence(test_input, chunks, raw_dir, package_dir, input_path)
splits = {"dev": dev_items, "test": test_items}

audit = pd.DataFrame([
    {"split": "dev", **correspondence_report(
        [it for it in items if it["evidence_source"] == "golden"], dev_items, chunks)},
    {"split": "test", **correspondence_report(test_input, test_items, chunks)},
])
audit.to_csv(ARTIFACTS / "golden_correspondence.csv", index=False)
display(audit)


## 문서 연결과 미연결 목록

`unanswerable`의 인용 근거는 다른 사업 문서의 예시이므로 질문 대상 문서의 정답 근거로 해석하지 않습니다.
이 보고서는 원래 인용한 근거의 소속만 연결합니다. 따라서 dev 질문 대상 30문서와 근거의 36 SHA는 다릅니다.


In [ ]:
documents = document_report(splits)
documents.to_csv(ARTIFACTS / "golden_documents.csv", index=False)
display(documents.groupby(["split", "outcome"]).size().unstack(fill_value=0))
display(documents[documents["outcome"] == "doc_unmapped"])


## 주제별 라벨 품질 — 검색 없이 dev · test 전체

각 행은 근거 항목 수와 결과별 개수, 모든 근거·일부 근거·어느 근거도 청크 라벨이 없는 질문 수입니다.
`multi_hop`과 `comparison`을 따로 표시합니다. 03의 핵심 사실 점검에서 빠지는 질문도 여기에는 포함합니다.
`document_rate`, `chunk_rate`의 분모는 각각 해당 행의 전체 근거 항목 수입니다.


In [ ]:
mapping = mapping_report(splits)
mapping.to_csv(ARTIFACTS / "golden_mapping.csv", index=False)
display(mapping)


## 평가 항목 점검 · 저장

dev와 YAML 항목만 저장합니다. YAML은 `evidence_source = "none"`, `doc_ids = []`, `evidence = []`로 표시합니다.
골든 항목에는 `doc_ids`와 각 근거의 본문·현재 `doc_id`·유일한 `chunk_id` 또는 사유를 저장합니다.
`golden_chunk_id`에는 원래 ID를 남겨 현재 라벨과 구분합니다.


In [ ]:
eval_set, report = check_eval_items(labelled_items)
write_jsonl(eval_set, ARTIFACTS / "eval_set.jsonl")
print(f"평가 데이터셋 {len(eval_set)}개 / 점검 {len(items)}개")
display(report["점검"].value_counts())
display(report[report["점검"] != "통과"])


## 고정 seed 42 — 라벨된 dev 근거 30쌍 대조

질문 ID·청크 ID·근거 본문 순으로 정렬한 라벨된 근거에서 최대 30항목을 비복원 추출합니다. 30쌍 미만이면 전부, 0쌍이면 빈 표를 표시합니다.
왼쪽은 골든 근거 본문, 오른쪽은 연결한 청크의 전체 `body`입니다. 가로로 나란히 읽으며 확인합니다.


In [ ]:
sample = labelled_sample(dev_items, chunks)
display(sample.style.format(escape="html").set_properties(**{
    "white-space": "pre-wrap", "text-align": "left", "vertical-align": "top",
}).set_table_styles([
    {"selector": "td", "props": [("min-width", "160px"), ("max-width", "540px")]},
]))
